In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ALEJANDRO MELGUIZO
# DATE: 8/18/2026
# TOPIC: new file to filter and clean data for analysis
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk
import statsmodels.api as sm
import statsmodels.formula.api as smf

### NAME SPACE: 
df_1: original cleaned data from previous project cleaning (see econometrics final project)

df_OA = data cleaned here, includes recleaned hispanic var, log wages, and not filtered to just include hispanics

df_hisp = df_OA filtered to only include hispanic

In [2]:
#set working directory
os.chdir("C:/Users/A.Melguizo001/Downloads/Immig, Wages, Latinos")

#preliminary coding
    #defining data frames
df_1 = pd.read_csv('gaston_df_v2.csv')
df_OA = df_1.copy()

## New var construction

In [3]:
#construcing a new var that measures roughly whether a person got their bachelor's or higher in the US
df_OA['US_educ'] = 0

hispan_educ_cases = [
    ((df_OA['educ_clean'] >= 3) & (df_OA['AGEATIMMIG'] <= 17), 1)
]

df_OA['US_educ'] = (
    pd.Series(np.nan, index = df_OA.index)
    .case_when(hispan_educ_cases)
    .fillna(0)
    .astype(int)
)

In [4]:
hispan_race_cases = [
    ((df_OA['RACE'] == 100), 0), #white
    ((df_OA['RACE'] == 200), 1), #black
    ((df_OA['RACE'] == 300), 2), #native american
    ((df_OA['RACE'].isin(range(650, 653))), 3), #asian / pacific islander
    ((df_OA['RACE'] == 700), 4),
    ((df_OA['RACE'].isin(range(801, 831))), 4),
    ((df_OA['RACE'] == 999), np.nan)
]

df_OA['race_clean'] = (
    pd.Series(np.nan, index = df_OA.index)
    .case_when(hispan_race_cases)
    .fillna(np.nan)
    .astype(int)
)

## Data Cleaning

In [5]:
#dropping hisp_clean to re-categorize
df_OA = df_OA.drop(columns= ['hisp_clean'])

#re-categorizing hispanic category
hispan_cases = [
    (df_OA['HISPAN'] == 0, 0),
    ((df_OA['HISPAN'] >= 100) & (df_OA['HISPAN'] <= 109), 1),
    (df_OA['HISPAN'] == 200, 2),
    (df_OA['HISPAN'] == 300, 3), 
    (df_OA['HISPAN'] == 400, 4),  #Dominicans
    (df_OA['HISPAN'] == 500, 5),
    ((df_OA['HISPAN'] == 610) | (df_OA['HISPAN'] == 611), 6),
    (df_OA['HISPAN'] == 612, 7),  #South American
    (df_OA['HISPAN'] == 600, 8),  #Other Hispanic
    ((df_OA['HISPAN'].isna()) | (df_OA['HISPAN'] == 902), 9)    #NA/IDK responses
]

df_OA['hisp_clean'] = (
    pd.Series(np.nan, index = df_OA.index)
    .case_when(hispan_cases)
    .fillna(9)
    .astype(int)
)


#making log wages for regression

df_OA = df_OA[
    (df_OA['incwage_clean'] > 0)
    ]

df_OA['log_wage'] = np.log(df_OA['incwage_clean'])

In [6]:
df_hisp = df_OA.copy()

#filter to only include:
#respondents who are: hispanic, born outside of the US and territories, have data for AGEATIMMIG, and have a non-0 wage.
df_hisp = df_hisp[
    (df_hisp['hisp_clean'] > 0) &
    (df_hisp['hisp_clean'] != 9) &
    (df_hisp['bpl_binary'] == 1) &
    (df_hisp['AGEATIMMIG'].notna())
]

In [7]:

# Massachusetts filter
df_hisp_MA = df_hisp[
    (df_hisp['STATEFIP'] == 25)
]

## Download clean csv

In [9]:
#download fully cleaned csv
df_OA.to_csv('gaston_OA.csv', index = False)

df_hisp.to_csv('gaston_1.csv', index=False)

df_hisp_MA.to_csv('gaston_1_MA.csv', index=False)